In [2]:
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
import pandas as pd
import csv

# CSV 파일 불러오기
with open('../csv/daily_economy_url.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    news = list(reader)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

articles = []
for idx, item in tqdm(enumerate(news), total=len(news)):
    url = item['url']
    title = item['title']

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # ✅ 매일경제 본문 div 선택
        container = soup.find('div', class_='news_cnt_detail_wrap', itemprop='articleBody')

        if container:
            # <p> 또는 <div> 내 텍스트 추출
            texts = [tag.get_text(strip=True) for tag in container.find_all(['p', 'div']) if tag.get_text(strip=True)]
            full_text = ' '.join(texts)
        else:
            full_text = ''

        articles.append({
            'url': url,
            'title': title,
            'content': full_text
        })

    except Exception as e:
        print(f"[{idx+1}] 오류 발생: {e}")
        articles.append({
            'url': url,
            'title': title,
            'content': ''
        })

print(f"총 수집된 기사 수: {len(articles)}")

# CSV로 저장
df = pd.DataFrame(articles)
df.to_csv("../csv/daily_economy_article.csv", index=False)

100%|██████████| 990/990 [03:43<00:00,  4.43it/s]

총 수집된 기사 수: 990


In [3]:
df.head()

,url,title,content
0,https://www.mk.co.kr/news/realestate/11323758,이제 좀 내리나 했더니...또 서울 집값만 뜀폭 키우는 이유가,사진 확대19일 서울 송파구 부동산의 모습. [사진 출처 = 연합뉴스] 사진 확대 ...
1,https://www.mk.co.kr/news/realestate/11323646,토허제도 못 말린 ‘똘똘한 한채’ 선호…강남·용산 집값 상승폭 또 올라,"부동산원 주간 아파트 가격동향서울 아파트값, 16주 연속 상승 부동산원 주간 아파트..."
2,https://www.mk.co.kr/news/realestate/11323555,"‘칸타빌’ 대원, 베트남 부동산 투자·개발 성과 잇따라","‘칸타빌’ 대원, 베트남 개발사업 주력다푹 신도시·안푸 칸타빌 등 사업 수행올 하반..."
3,https://www.mk.co.kr/news/realestate/11323507,"높이규제지역 공공기여 줄이고, 지하철역 250m 이내 준주거 종상향",서울시 2030 도시·주거 기본계획 변경높이규제 지역 용적률 상향시늘어나는 정도에 ...
4,https://www.mk.co.kr/news/realestate/11323320,“강남도 용산도 아닌데 국평 30억 찍었다”…토허제 비껴간 이 동네 ‘후끈’,신고가 새로 쓰는 한강변흑석동 국평도 이젠 ‘30억’아크로리버하임 84㎡31.2억 ...


In [4]:
df['content']

0      사진 확대19일 서울 송파구 부동산의 모습. [사진 출처 = 연합뉴스] 사진 확대 ...
1      부동산원 주간 아파트 가격동향서울 아파트값, 16주 연속 상승 부동산원 주간 아파트...
2      ‘칸타빌’ 대원, 베트남 개발사업 주력다푹 신도시·안푸 칸타빌 등 사업 수행올 하반...
3      서울시 2030 도시·주거 기본계획 변경높이규제 지역 용적률 상향시늘어나는 정도에 ...
4      신고가 새로 쓰는 한강변흑석동 국평도 이젠 ‘30억’아크로리버하임 84㎡31.2억 ...
                             ...                        
985    尹 올해 공공분양 9만호 공급 발표했지만11월까지 4700호 공급... 서울은 ‘0...
986    사진 확대[사진 출처 = 연합뉴스] 사진 확대 내년 주요 건설사의 확정된 민간 아파...
987    사진 확대서울 송파구 잠실주공 5단지 전경.  [사진출처=연합뉴스] 사진 확대 전용...
988    민간 예정물량 25년來 최저수도권은 1년새 30% 급감집값 양극화 더 부채질할듯 민...
989    서울원아이파크·평촌자이높은 경쟁률로 마감했지만상당수 물량 미계약 발생에통장필요없는 ...
Name: content, Length: 990, dtype: object

In [5]:
import pandas as pd
df = pd.read_csv('../csv/daily_economy_article.csv')

# 크롤링한 뉴스데이터에서 해당 구가 있는지 확인
dvsn_list = [
    '강남구', '강동구', '강북구', '강서구', '관악구',
    '광진구', '구로구', '금천구', '노원구', '도봉구',
    '동대문구', '동작구', '마포구', '서대문구', '서초구',
    '성동구', '성북구', '송파구', '양천구', '영등포구',
    '용산구', '은평구', '종로구', '중구', '중랑구'
]

cnt = 0
for content in df['content']:
    if pd.isna(content):
        continue
    for dvsn in dvsn_list:
        if dvsn in content:
            cnt += 1
            break  # 중복 방지: 한 기사에 여러 자치구가 있어도 한 번만 셈

print(f"자치구가 언급된 기사 수: {cnt}개")

자치구가 언급된 기사 수: 450개


In [8]:
from collections import defaultdict

district_count = defaultdict(int)

for content in df['content']:
    if pd.isna(content):
        continue
    for dvsn in dvsn_list:
        if dvsn in content:
            district_count[dvsn] += 1

# 결과 보기
for dvsn, count in district_count.items():
    print(f"{dvsn}: {count}건")

강남구: 161건
마포구: 43건
서초구: 175건
성동구: 64건
송파구: 156건
용산구: 121건
강동구: 57건
강북구: 28건
동작구: 26건
양천구: 38건
영등포구: 35건
은평구: 34건
중구: 32건
중랑구: 24건
구로구: 41건
관악구: 26건
광진구: 35건
동대문구: 23건
종로구: 27건
노원구: 47건
도봉구: 21건
강서구: 30건
성북구: 26건
금천구: 23건
서대문구: 9건
